In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "enviroment_bj").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("project_root:", ROOT)

project_root: C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl


In [2]:
from enviroment_bj import BlackjackEnvironment, BlackjackConfig, ObservationConfig, StartStateConfig
from loss import BellmanLossConfig, LossPhaseWeightConfig
from model.agents import DuelingRecurrentDoubleDQN
from training import (
    train_model,
    ReplayBufferConfig,
    EpsilonScheduleConfig,
    DualEpsilonConfig,
    NStepConfig,
    OptimizationConfig,
    TargetUpdateConfig,
    EvaluationConfig,
    CheckpointConfig,
    PrintConfig,
    TrainerConfig,
    TrainingPipelineConfig,
)
from model.encoder import BlackjackObservationEncoder, EncoderConfig
from model.agents import DuelingRecurrentDoubleDQN, AgentNetworkConfig
from copy import deepcopy

In [5]:
# ============================================================
# OBSERVATION: realista para shoe desconocido
# ============================================================

observation_config = ObservationConfig(
    profile="table_realistic_unknown_progress",
    obs_include_table_rules=True,
    obs_include_visible_rules_only=True,
    obs_include_hidden_rules=False,
    obs_include_decision_phase=True,
    obs_include_available_bet_multipliers=True,
    obs_current_hand_mode="table_raw",
    obs_include_other_player_hands=True,
    obs_include_current_bet=True,
    obs_include_betting_context=True,
    obs_include_hand_context=True,
    obs_include_insurance_context=True,
    obs_include_temporal_context=True,
    obs_include_hands_since_shuffle=False,
    obs_include_estimated_shoe_progress=False,
    obs_include_last_hand_outcome=False,
    obs_include_recent_actions=False,
    obs_recent_actions_window=8,
    obs_include_observed_cards_history=True,
    obs_observed_cards_mode="rank_counts",
    obs_recent_cards_window=32,
    obs_reset_history_on_shuffle=True,
    obs_include_exact_shoe_composition=False,
    obs_include_discard_summary=True,
    obs_include_n_decks=False,
    obs_include_shoe_penetration_rule=False,
)

# ============================================================
# START STATE
# ============================================================

start_state_config = StartStateConfig(
    mode="unknown_progress",
    min_burned_rounds=10,
    max_burned_rounds=60,
    clear_visible_histories_after_burn=True,
    hide_reshuffle_progress_from_observation=True,
)

# ============================================================
# TABLE CONFIG
# ============================================================

blackjack_config = BlackjackConfig(
    n_decks=8,
    shoe_penetration=0.75,
    use_cut_card=True,
    dealer_hits_soft_17=False,          # S17
    blackjack_payout=1.5,               # 3:2
    dealer_peeks_for_blackjack=True,
    double_allowed_on="any_two_cards",
    double_after_split_allowed=True,
    double_split_aces_allowed=False,
    split_rule="same_value",
    max_hands_after_split=2,
    max_split_depth_per_hand=1,
    resplit_aces_allowed=False,
    hit_split_aces_allowed=False,
    surrender_allowed=False,
    insurance_allowed=True,
    six_card_charlie_enabled=True,
    base_bet=1.0,
    bet_multipliers=(1,),
    strict_shoe_validation=False,
    observation=observation_config,
    observation_mode=None,
    expose_shoe_composition=False,
    visible_shoe_change=True,
    exogenous_cards=True,
)

# ============================================================
# ENVS
# ============================================================

base_seed = 35
num_envs = 3

penetrations = [0.65, 0.75, 0.85]
exogenous_profiles = ["passive", "normal", "aggressive"]

envs = []
for i, (penetration, exo_profile) in enumerate(zip(penetrations, exogenous_profiles)):
    cfg = deepcopy(blackjack_config)
    cfg.shoe_penetration = penetration
    cfg.exogenous_cards_profile = exo_profile

    env = BlackjackEnvironment(
        config=cfg,
        seed=base_seed + i,
        start_state=start_state_config,
    )
    envs.append(env)

In [20]:
# ============================================================
# MODEL
# ============================================================

encoder_config = EncoderConfig(
    profile="table_realistic_unknown_progress",

    # modulos principales
    encode_rules=True,
    encode_betting_context=True,
    encode_other_hands=True,
    encode_temporal=True,
    encode_observed_history=True,
    encode_discard_summary=True,

    # extras que ahora sí quieres por defecto
    encode_recent_actions=True,
    encode_exact_shoe=False,          # deja esto en False para mesa realista
    encode_action_mask_features=False,

    # detalles de codificacion
    card_encoding="one_hot_rank",
    history_encoding="rank_counts",
    normalize_counts=True,
    use_visible_table_rules_only=True,

    # limites / ventanas
    max_current_hand_cards=12,
    max_cards_per_hand=12,
    max_other_hands=4,
    max_recent_actions=5,
    max_recent_cards=32,
    max_recent_discard_cards=10,
)

encoder = BlackjackObservationEncoder(config=encoder_config)

model_config = AgentNetworkConfig.for_architecture(
    "dueling_recurrent",
    encoder_profile=encoder_config.profile,
    activation="relu",
    use_layer_norm=True,
    dropout=0.0,
    projection_dim=256,
    recurrent_hidden_dim=256,
    recurrent_num_layers=1,
    recurrent_type="gru",
    value_hidden_dim=128,
    advantage_hidden_dim=128,
    use_phase_adapters=True,
    use_module_gating=True,
)

model = DuelingRecurrentDoubleDQN(
    config=model_config,
    encoder=encoder,
)

# ============================================================
# REPLAY BUFFER
# ============================================================

replay_buffer_config = ReplayBufferConfig(
    capacity=135_000,
    batch_size=64,
    warmup_size=8_000,
    sequence_length=16,
    min_sequence_length=8,
)

# ============================================================
# EPSILON DUAL
# Cuarta corrida: más lenta la caída para evitar congelar
# demasiado pronto betting y playing
# ============================================================

dual_epsilon_config = DualEpsilonConfig(
    betting=EpsilonScheduleConfig(
        start=1.0,
        end=0.10,
        decay_steps=260_000,
        evaluation_epsilon=0.0,
    ),
    playing=EpsilonScheduleConfig(
        start=1.0,
        end=0.05,
        decay_steps=180_000,
        evaluation_epsilon=0.0,
    ),
)

# ============================================================
# N-STEP
# ============================================================

n_step_config = NStepConfig(
    enabled=True,
    n_steps=3,
)

# ============================================================
# OPTIMIZATION
# ============================================================

optimization_config = OptimizationConfig(
    optimizer="adam",
    learning_rate=1e-4,
    weight_decay=1e-5,
    scheduler="none",
    scheduler_step_size=1000,
    scheduler_gamma=0.99,
    gradient_clipping=True,
    max_grad_norm=5.0,
)

# ============================================================
# TARGET NETWORK
# ============================================================

target_update_config = TargetUpdateConfig(
    mode="soft",
    hard_update_interval=2000,
    soft_tau=0.002,
)

# ============================================================
# EVALUATION
# Subimos un poco la robustez de la validación
# ============================================================

evaluation_config = EvaluationConfig(
    enabled=True,
    every_n_epochs=4,
    num_rounds=1200,
    max_decisions=100_000,
)

# ============================================================
# CHECKPOINTS
# ============================================================

checkpoint_config = CheckpointConfig(
    directory="training_checkpoints/dueling_unknown_progress_realistic_table_run4",
    save_latest=True,
    save_best_eval=True,
    save_periodic=True,
    periodic_interval_updates=5000,
    best_metric_name="ev_per_1000_hands",
    maximize_best_metric=True,
)

# ============================================================
# PRINT CONFIG
# ============================================================

print_config = PrintConfig(
    enable=True,
    print_run_summary=True,
    print_warmup_interval=1000,
    print_update_interval=400,
    print_collection_interval=800,
    print_epoch_header=True,
    print_epoch_summary=True,
    print_eval_summary=True,
    include_segment_details=False,
)

# ============================================================
# LOSS
# ============================================================

loss_config = BellmanLossConfig(
    gamma=0.99,
    loss_type="huber",
    validate_current_actions=True,
    validate_next_action_mask=True,
    allow_terminal_without_legal_next_action=True,
    phase_weights=LossPhaseWeightConfig(
        enabled=True,
        betting_weight=1.5,
        playing_weight=1.0,
    ),
)

# ============================================================
# TRAINER
# Un poco más de experiencia por época para esta corrida
# ============================================================

trainer_config = TrainerConfig(
    total_epochs=35,
    env_steps_per_epoch=3_500,
    train_frequency=4,
    updates_per_train_step=1,
    max_updates_per_epoch=None,
    device="cpu",
    seed=18,
    reset_hidden_on_round_end=False,
    sequence_end_on_done=False,
    flush_partial_sequences_at_epoch_end=True,
    loss=loss_config,
)

# ============================================================
# PIPELINE
# ============================================================

pipeline_config = TrainingPipelineConfig(
    trainer=trainer_config,
    replay_buffer=replay_buffer_config,
    epsilon=dual_epsilon_config,
    n_step=n_step_config,
    optimization=optimization_config,
    target_update=target_update_config,
    evaluation=evaluation_config,
    checkpoints=checkpoint_config,
    prints=print_config,
)

# ============================================================
# TRAIN
# ============================================================

result = train_model(
    envs=envs,
    model=model,
    pipeline_config=pipeline_config,
)

BLACKJACK RL RUN
  Model      : arch=dueling_recurrent | recurrent=gru | encoder=table_realistic_unknown_progress | obs=table_realistic_unknown_progress | start=unknown_progress
  Runtime    : device=cpu | epochs=35 | envs=3 | steps/epoch=10500 | updates/epoch~=2625 | params=971,542
  Optim      : optimizer=adam | lr=1.00e-04 | loss=huber | gamma=0.9900 | grad_clip=True(5.00)
  Replay     : warmup=8000 | capacity=135000 | batch=64 | seq_len=24 | min_seq_len=12
  Explore    : eps_bet=1.000->0.100 (decay 260000) | eps_play=1.000->0.050 (decay 180000) | target=soft | interval=2000 | tau=0.0020
  Extras     : n_step=True(3) | phase_loss_w=True (bet 1.50, play 1.00) | phase_adapters=True | module_gating=True
  Eval / CKPT: eval_rounds=1200 | eval_decisions=100000 | checkpoints=training_checkpoints\dueling_unknown_progress_realistic_table_run4
  Table      : decks=8 | pen=0.65 | S17=True | payout=1.50 | double=any_two_cards | split=same_value | DAS=True

=== Epoch 1/35 ===
[Warmup] buffer 10

KeyboardInterrupt: 

## Segunda corrida para mejorar beting 

In [9]:
# ============================================================
# OBSERVATION: realista para shoe desconocido
# ============================================================

observation_config = ObservationConfig(
    profile="table_realistic_unknown_progress",
    obs_include_table_rules=True,
    obs_include_visible_rules_only=True,
    obs_include_hidden_rules=False,
    obs_include_decision_phase=True,
    obs_include_available_bet_multipliers=True,
    obs_current_hand_mode="table_raw",
    obs_include_other_player_hands=True,
    obs_include_current_bet=True,
    obs_include_betting_context=True,
    obs_include_hand_context=True,
    obs_include_insurance_context=True,
    obs_include_temporal_context=True,
    obs_include_hands_since_shuffle=False,
    obs_include_estimated_shoe_progress=False,
    obs_include_last_hand_outcome=False,
    obs_include_recent_actions=False,
    obs_recent_actions_window=8,
    obs_include_observed_cards_history=True,
    obs_observed_cards_mode="rank_counts",
    obs_recent_cards_window=32,
    obs_reset_history_on_shuffle=True,
    obs_include_exact_shoe_composition=False,
    obs_include_discard_summary=True,
    obs_include_n_decks=False,
    obs_include_shoe_penetration_rule=False,
)

# ============================================================
# START STATE
# ============================================================

start_state_config = StartStateConfig(
    mode="unknown_progress",
    min_burned_rounds=10,
    max_burned_rounds=60,
    clear_visible_histories_after_burn=True,
    hide_reshuffle_progress_from_observation=True,
)

# ============================================================
# TABLE CONFIG
# ============================================================

blackjack_config = BlackjackConfig(
    n_decks=8,
    shoe_penetration=0.75,
    use_cut_card=True,
    dealer_hits_soft_17=False,          # S17
    blackjack_payout=1.5,               # 3:2
    dealer_peeks_for_blackjack=True,
    double_allowed_on="any_two_cards",
    double_after_split_allowed=True,
    double_split_aces_allowed=False,
    split_rule="same_value",
    max_hands_after_split=2,
    max_split_depth_per_hand=1,
    resplit_aces_allowed=False,
    hit_split_aces_allowed=False,
    surrender_allowed=False,
    insurance_allowed=True,
    six_card_charlie_enabled=True,
    base_bet=1.0,
    bet_multipliers=(1,2,3,4),
    strict_shoe_validation=False,
    observation=observation_config,
    observation_mode=None,
    expose_shoe_composition=False,
    visible_shoe_change=True,
    exogenous_cards=True,
)

# ============================================================
# ENVS
# ============================================================

base_seed = 99
num_envs = 3

penetrations = [0.65, 0.75, 0.85]
exogenous_profiles = ["passive", "normal", "aggressive"]

envs = []
for i, (penetration, exo_profile) in enumerate(zip(penetrations, exogenous_profiles)):
    cfg = deepcopy(blackjack_config)
    cfg.shoe_penetration = penetration
    cfg.exogenous_cards_profile = exo_profile

    env = BlackjackEnvironment(
        config=cfg,
        seed=base_seed + i,
        start_state=start_state_config,
    )
    envs.append(env)

In [10]:
dual_epsilon_config = DualEpsilonConfig(
    betting=EpsilonScheduleConfig(
        start=0.85,
        end=0.08,
        decay_steps=150_000,
        evaluation_epsilon=0.0,
    ),
    playing=EpsilonScheduleConfig(
        start=0.08,
        end=0.02,
        decay_steps=35_000,
        evaluation_epsilon=0.0,
    ),
)

loss_config = BellmanLossConfig(
    gamma=0.99,
    loss_type="huber",
    validate_current_actions=True,
    validate_next_action_mask=True,
    allow_terminal_without_legal_next_action=True,
    phase_weights=LossPhaseWeightConfig(
        enabled=True,
        betting_weight=2.0,
        playing_weight=0.5,
    ),
)


replay_buffer_config = ReplayBufferConfig(
    capacity=135_000,
    batch_size=64,
    warmup_size=8_000,
    sequence_length=16,
    min_sequence_length=8,
)

# ============================================================
# TRAINER
# Un poco más de experiencia por época para esta corrida
# ============================================================

trainer_config = TrainerConfig(
    total_epochs=65,
    env_steps_per_epoch=3_500,
    train_frequency=4,
    updates_per_train_step=1,
    max_updates_per_epoch=None,
    device="cpu",
    seed=18,
    reset_hidden_on_round_end=False,
    sequence_end_on_done=False,
    flush_partial_sequences_at_epoch_end=True,
    loss=loss_config,
)

# ============================================================
# PIPELINE
# ============================================================

pipeline_config = TrainingPipelineConfig(
    trainer=trainer_config,
    replay_buffer=replay_buffer_config,
    epsilon=dual_epsilon_config,
    n_step=n_step_config,
    optimization=optimization_config,
    target_update=target_update_config,
    evaluation=evaluation_config,
    checkpoints=checkpoint_config,
    prints=print_config,
)

# ============================================================
# TRAIN
# ============================================================
checkpoint_path = r"C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl\outputs\modelos run1 (solo playing- 30 epocas )\latest.pt"

result = train_model(
    envs=envs,
    model=model,
    pipeline_config=pipeline_config,
    resume=True,
    resume_checkpoint_path=checkpoint_path,
)

BLACKJACK RL RUN
  Model      : arch=dueling_recurrent | recurrent=gru | encoder=table_realistic_unknown_progress | obs=table_realistic_unknown_progress | start=unknown_progress
  Runtime    : device=cpu | epochs=65 | envs=3 | steps/epoch=10500 | updates/epoch~=2625 | params=972,566
  Optim      : optimizer=adam | lr=1.00e-04 | loss=huber | gamma=0.9900 | grad_clip=True(5.00)
  Replay     : warmup=8000 | capacity=135000 | batch=64 | seq_len=16 | min_seq_len=8
  Explore    : eps_bet=0.850->0.080 (decay 150000) | eps_play=0.080->0.020 (decay 35000) | target=soft | interval=2000 | tau=0.0020
  Extras     : n_step=True(3) | phase_loss_w=True (bet 2.00, play 0.50) | phase_adapters=True | module_gating=True
  Eval / CKPT: eval_rounds=1200 | eval_decisions=100000 | checkpoints=training_checkpoints\dueling_unknown_progress_realistic_table_run4
  Table      : decks=8 | pen=0.65 | S17=True | payout=1.50 | double=any_two_cards | split=same_value | DAS=True

=== Epoch 31/65 ===
[Warmup] buffer 100

KeyboardInterrupt: 